# SLC46A1 engraftment, step 1: zero-shot risk scoring of the primary humanising substitutions

Scores every pharmacophore position at which chicken PCFT and human PCFT differ, as the
ESM-2 marginal probability of the human residue in the chicken wild-type background, and
flags those below the operative threshold of 1e-3.

Every cell aborts on a failed precondition. Results are written to Drive, not to the
console, so the run can be left unattended.

In [ ]:
import os, sys, json, hashlib, datetime

from google.colab import drive
drive.mount('/content/drive')

PROJECT = '/content/drive/MyDrive/MIT/All Project Files/SLC46A1 engraftment'
OUT     = os.path.join(PROJECT, 'esm2_step1')

if not os.path.isdir(PROJECT):
    raise SystemExit('FATAL project folder not found: %s' % PROJECT)
os.makedirs(OUT, exist_ok=True)

REQUIRED = ['data/pharmacophore_position_set.tsv', 'structures/7BC6_to_human.json']
missing = [f for f in REQUIRED if not os.path.exists(os.path.join(PROJECT, f))]
if missing:
    raise SystemExit('FATAL missing input files in %s: %s' % (PROJECT, missing))

RUNLOG = os.path.join(OUT, 'run.log')

def log(msg):
    line = '%s  %s' % (datetime.datetime.utcnow().isoformat(timespec='seconds'), msg)
    with open(RUNLOG, 'a') as fh:
        fh.write(line + chr(10))

log('start, project=%s' % PROJECT)


In [ ]:
import shutil, subprocess

if shutil.which('nvidia-smi') is None:
    raise SystemExit('FATAL this runtime has no GPU. Set Runtime > Change runtime '
                     'type > T4 GPU (or better), then run from the first cell again.')
if subprocess.run(['nvidia-smi'], capture_output=True).returncode != 0:
    raise SystemExit('FATAL nvidia-smi is present but failing, so the runtime has no '
                     'usable GPU. Change the runtime type and run from the first cell.')

subprocess.run([sys.executable, '-m', 'pip', '-q', 'install',
                'torch', 'transformers', 'biopython'], check=True)

import torch
from transformers import AutoTokenizer, EsmForMaskedLM

if not torch.cuda.is_available():
    raise SystemExit('FATAL torch cannot see the GPU after install')
log('torch %s, device %s' % (torch.__version__, torch.cuda.get_device_name(0)))


In [ ]:
import urllib.request

def fetch(url, path):
    if os.path.exists(path) and os.path.getsize(path) > 0:
        return path
    urllib.request.urlretrieve(url, path)
    if os.path.getsize(path) == 0:
        raise SystemExit('FATAL empty download: %s' % url)
    return path

SEQ = os.path.join(OUT, 'seq')
os.makedirs(SEQ, exist_ok=True)
fetch('https://rest.uniprot.org/uniprotkb/Q96NT5.fasta', os.path.join(SEQ, 'Q96NT5.fasta'))
fetch('https://www.rcsb.org/fasta/entry/7BC6', os.path.join(SEQ, '7BC6.fasta'))

human = ''.join(l.strip() for l in open(os.path.join(SEQ, 'Q96NT5.fasta'))
                if not l.startswith('>'))

ent = {}
for rec in open(os.path.join(SEQ, '7BC6.fasta')).read().split('>'):
    if rec.strip():
        parts = rec.strip().split(chr(10))
        ent[parts[0].split('|')[0]] = ''.join(parts[1:])
chicken = ent['7BC6_1'][:473]

if len(human) != 459:
    raise SystemExit('FATAL Q96NT5 length %d, expected 459' % len(human))
if len(chicken) != 473:
    raise SystemExit('FATAL chicken native length %d, expected 473' % len(chicken))
log('sequences ok: human %d, chicken %d' % (len(human), len(chicken)))


In [ ]:
path = os.path.join(PROJECT, 'data', 'pharmacophore_position_set.tsv')
rows = [l.rstrip(chr(10)).split(chr(9)) for l in open(path)]
col = {k: i for i, k in enumerate(rows[0])}

primaries = []
for r in rows[1:]:
    if r[col['differs']] != '1':
        continue
    hp, cp = int(r[col['human_pos']]), int(r[col['chicken_pos']])
    haa, caa = r[col['human_aa']], r[col['chicken_aa']]
    if chicken[cp - 1] != caa:
        raise SystemExit('FATAL chicken %d is %s in sequence, %s in table'
                         % (cp, chicken[cp - 1], caa))
    if human[hp - 1] != haa:
        raise SystemExit('FATAL human %d is %s in sequence, %s in table'
                         % (hp, human[hp - 1], haa))
    primaries.append({'chicken_pos': cp, 'chicken_aa': caa,
                      'human_pos': hp, 'human_aa': haa})

if not primaries:
    raise SystemExit('FATAL no divergent positions found in the position set')
log('primary humanising substitutions to score: %d' % len(primaries))


In [ ]:
THRESHOLD = 1e-3
MODEL = 'facebook/esm2_t33_650M_UR50D'

tok = AutoTokenizer.from_pretrained(MODEL)
model = EsmForMaskedLM.from_pretrained(MODEL).eval().cuda()

enc = tok(chicken, return_tensors='pt')
ids = enc['input_ids'].cuda()
offset = 1

with torch.no_grad():
    probs_wt = torch.softmax(model(ids).logits[0], dim=-1)

for p in primaries:
    i = p['chicken_pos'] - 1 + offset
    masked = ids.clone()
    masked[0, i] = tok.mask_token_id
    with torch.no_grad():
        pr = torch.softmax(model(masked).logits[0, i], dim=-1)
    hid = tok.convert_tokens_to_ids(p['human_aa'])
    cid = tok.convert_tokens_to_ids(p['chicken_aa'])
    p['p_human_masked'] = float(pr[hid])
    p['p_chicken_masked'] = float(pr[cid])
    p['p_human_wt_background'] = float(probs_wt[i, hid])
    p['high_risk'] = bool(p['p_human_wt_background'] < THRESHOLD)
    log('%s%d->%s  p_wt=%.3e  p_masked=%.3e  high_risk=%s'
        % (p['chicken_aa'], p['chicken_pos'], p['human_aa'],
           p['p_human_wt_background'], p['p_human_masked'], p['high_risk']))


In [ ]:
tsv = os.path.join(OUT, 'primary_substitution_risk.tsv')
keys = ['chicken_pos', 'chicken_aa', 'human_pos', 'human_aa',
        'p_human_wt_background', 'p_human_masked', 'p_chicken_masked', 'high_risk']
with open(tsv, 'w') as fh:
    fh.write(chr(9).join(keys) + chr(10))
    for p in primaries:
        fh.write(chr(9).join(str(p[k]) for k in keys) + chr(10))

meta = {'model': MODEL,
        'threshold': THRESHOLD,
        'n_primary': len(primaries),
        'n_high_risk': sum(1 for p in primaries if p['high_risk']),
        'chicken_sha256': hashlib.sha256(chicken.encode()).hexdigest(),
        'human_sha256': hashlib.sha256(human.encode()).hexdigest(),
        'finished_utc': datetime.datetime.utcnow().isoformat(timespec='seconds')}
with open(os.path.join(OUT, 'run_meta.json'), 'w') as fh:
    json.dump(meta, fh, indent=2)

if not os.path.exists(tsv) or os.path.getsize(tsv) < 50:
    raise SystemExit('FATAL results file missing or too small: %s' % tsv)
log('wrote %s (%d bytes) and run_meta.json; high risk %d of %d'
    % (tsv, os.path.getsize(tsv), meta['n_high_risk'], meta['n_primary']))
